# Categorical Manipulation

### Loading Libraries

In [ ]:
# Numerical Computing
import numpy as np

# Data Manipulation
import pandas as pd

# PyArrow
import pyarrow as pa

# Data Visualisation
import seaborn as sns
import matplotlib.pyplot as plt

# OS
import io
import requests
from io import StringIO

# Numba
from numba import jit

# Collections
import collections

# Categorical Boosting
import catboost as cb

# Scikit-Learn
from sklearn import set_config
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

#### Retrieving Data

In [ ]:
pd.set_option('display.max_rows', 10) 

In [ ]:
# url = 'https://github.com/mattharrison/datasets/raw/master/data/'\
#     'siena2018-pres.csv'

In [ ]:
url = 'https://github.com/mattharrison/datasets/raw/master/data/' \
      'vehicles.csv.zip'

In [ ]:
df = pd.read_csv(url, engine='pyarrow', dtype_backend='pyarrow')

In [ ]:
make = df.make

In [ ]:
make

#### Frequency Counts

In [ ]:
make.value_counts()

In [ ]:
make.shape, make.nunique()

#### Beneficts of Categories

In [ ]:
cat_make = make.astype('category')

In [ ]:
make.memory_usage(deep=True)

In [ ]:
cat_make.memory_usage(deep=True)

In [ ]:
%%timeit
cat_make.str.upper()

In [ ]:
%%timeit
make.str.upper()

In [ ]:
old_make = make.astype(str)

In [ ]:
%%timeit
old_make.str.upper()

#### Conversion to Ordinal Categories

In [ ]:
make_type = pd.CategoricalDtype(
    categories=sorted(make.unique()), ordered=True)

In [ ]:
ordered_make = make.astype(make_type)

ordered_make

In [ ]:
(make
    .astype('category')
    .cat.as_ordered()
)

In [ ]:
ordered_make.max()

In [ ]:
cat_make.max()

In [ ]:
ordered_make.sort_values()

#### The `.cat` Accesor

In [ ]:
cat_make.cat.rename_categories(
    [c.lower() for c in cat_make.cat.categories])

In [ ]:
ordered_make.cat.rename_categories(
    {c:c.lower() for c in ordered_make.cat.categories})

In [ ]:
ordered_make.cat.reorder_categories(
    sorted(cat_make.cat.categories, key=str.lower))

#### Category Gotchas

In [ ]:
(ordered_make
    .head(100)
    .value_counts()
)

In [ ]:
(cat_make
    .head(100)
    .groupby(cat_make.head(100), observed=False)
    .first()
)

In [ ]:
(make
    .head(100)
    .groupby(make.head(100))
    .first()
)

In [ ]:
(cat_make
    .head(100)
    .groupby(cat_make.head(100), observed=True)
    .first()
)

In [ ]:
ordered_make.iloc[0]

In [ ]:
ordered_make.iloc[[0]]

#### Generalization

In [ ]:
def generalize_topn(ser, n=5, other='Other'):
    topn = ser.value_counts().index[:n]
    if isinstance(ser.dtype, pd.CategoricalDtype):
        ser = ser.cat.set_categories(
            topn.set_categories(list(topn) + [other]))
    return ser.where(ser.isin(topn), other)

In [ ]:
cat_make.pipe(generalize_topn, n=20, other='NA')

In [ ]:
def generalize_mapping(ser, mapping, default):
    seen = None
    res = ser.astype(str)
    for old, new in mapping.items():
        mask = ser.str.contains(old)
        if seen is None:
            seen = mask
        else:
            seen |= mask
        res = res.where(~mask, new)
    res = res.where(seen, default)
    return res.astype('category')

In [ ]:
generalize_mapping(cat_make, {
    'Ford':'US', 'Tesla':'US',
    'Chevrolet':'US', 'Dodge':'US',
    'Oldsmobile':'US', 'Plymouth':'US',
    'BMW':'German'}, 'Other')

# Dataframes

#### A Simple Python Version

In [ ]:
df = {
    'index': [0, 1, 2],
    'cols': [
        {'name':'growth',
         'data': [.5, .7, 1.2]},
        {'name':'Name',
         'data':['Paul', 'George', 'Ringo']},
    ]
}

In [ ]:
def get_row(df, idx):
    results = []
    value_idx = df['index'].index(idx)
    for col in df['cols']:
        results.append(col['data'][value_idx])
    return results

In [ ]:
get_row(df, 1)

In [ ]:
def get_col(df, name):
    for col in df['cols']:
        if col['name'] == name:
            return col['data']

In [ ]:
get_col(df, 'Name')

#### Dataframes

In [ ]:
df = pd.DataFrame({
    'growth': [.5, .7, 1.2],
    'Name': ['Paul', 'George', 'Ringo']})

In [ ]:
print(df)

In [ ]:
df.iloc[2]

In [ ]:
df['Name']

In [ ]:
type(df['Name'])

In [ ]:
df['Name'].str.lower()

#### Construction

In [ ]:
print(pd.DataFrame([
    {'growth':.5, 'Name': 'Paul'},
    {'growth':.7, 'Name': 'George'},
    {'growth':1.2, 'Name': 'Ringo'}]))

In [ ]:
csv_file = StringIO("""growth, Name
    .5, Paul
    .7, George
    1.2, Ringo
""")

In [ ]:
print(pd.read_csv(csv_file, dtype_backend='pyarrow', engine='pyarrow'))

In [ ]:
np.random.seed(42)

print(pd.DataFrame(np.random.randn(10, 3),
                  columns=['a', 'b', 'c']))

#### Dataframe Axis

In [ ]:
df.axes

In [ ]:
df.sum(axis=0)

In [ ]:
df.sum(axis=1)

In [ ]:
df.sum(axis='index')

In [ ]:
# df.sum(axis=1)

In [ ]:
df.axes[0]

In [ ]:
df.axes[1]

In [ ]:
df = pd.DataFrame({'Score1': [None, None],
                   'Score2': [85, 90]})

In [ ]:
print(df)

In [ ]:
df.apply(np.sum, axis=0)

# Similarities with Series & DataFrames

### Getting Data

In [ ]:
url = 'https://en.wikipedia.org/wiki/'\
'Historical_rankings_of_presidents_of_the_United_States'

In [ ]:
# pres_dfs = pd.read_html(url, dtype_backend='pyarrow')

In [ ]:
headers = {

    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"

}

response = requests.get(url, headers=headers)

response.raise_for_status()

In [ ]:
pres_dfs = pd.read_html(StringIO(response.text), dtype_backend="pyarrow")

In [ ]:
pres_dfs

In [ ]:
df = pres_dfs[2]

In [ ]:
df

In [ ]:
for i, table in enumerate(pres_dfs):
    print(i, table.columns)

In [ ]:
df = pres_dfs[0]  

In [ ]:
# (df
#     .iloc[1:-1]
#     .rename(columns={'Political party': 'Party'})
#     .assign(Party=lambda df_:df_
#         .Party
#         .str.replace(r'\[.*\]', '')
#         .astype('category'))
# )

In [ ]:
(df
    .iloc[1:-1]
    .rename(columns={'Political party': 'Party'})
    .assign(Party=lambda df_: df_['Party']
        .str.replace(r'\[.*?\]', '', regex=True)
        .astype('category'))
)

In [ ]:
df.dtypes

#### Part II

In [ ]:
url = 'https://github.com/mattharrison/datasets/raw/master/data/siena2018-pres.csv'

In [ ]:
df = pd.read_csv(url, index_col=0, dtype_backend='pyarrow')

In [ ]:
print(df)

In [ ]:
df.dtypes

In [ ]:
def tweak_siena_pres(df):
    rename_map = {
        "Bg": "Background",
        "PL": "Party leadership",
        "CAb": "Communication ability",
        "RC": "Relations with Congress",
        "CAp": "Court appointments",
        "HE": "Handling of economy",
        "L": "Luck",
        "AC": "Ability to compromise",
        "WR": "Willing to take risks",
        "EAp": "Executive appointments",
        "OA": "Overall ability",
        "Im": "Imagination",
        "DA": "Domestic accomplishments",
        "Int": "Integrity",
        "EAb": "Executive ability",
        "FPA": "Foreign policy accomplishments",
        "LA": "Leadership ability",
        "IQ": "Intelligence",
        "AM": "Avoid crucial mistakes",
        "EV": "Experts view",
        "O": "Overall",
    }

    score_cols = [v.replace(" ", "_") for v in rename_map.values()]

    def int64_to_uint8(df_):
        cols = df_.select_dtypes(include=["int64", "int64[pyarrow]"]).columns
        return df_.astype({col: "uint8[pyarrow]" for col in cols})

    return (
        df
        .rename(columns={"Seq.": "Seq"})
        .rename(columns={k: v.replace(" ", "_") for k, v in rename_map.items()})
        .astype({"Party": "category"})
        .pipe(int64_to_uint8)
        .assign(
            Average_rank=lambda df_: (
                df_[score_cols]
                .astype("uint8")
                .sum(axis=1)
                .rank(method="dense")
                .astype("uint8[pyarrow]")
            ),
            Quantile=lambda df_: pd.qcut(
                df_.Average_rank,
                q=4,
                labels=["1st", "2nd", "3rd", "4th"],
                duplicates="drop"
            )
        )
    )

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10), dpi=600)

g = sns.heatmap((tweak_siena_pres(df)
    .set_index('President')
    .loc[:, 'Background':'Average_rank']
    .astype('uint8')
                ), annot=True, cmap='viridis', ax=ax)

g.set_xticklabels(g.get_xticklabels(), rotation=45, fontsize=8, ha='right')

_ = plt.title('Presidential Ranking')

#### Viewing Data

In [ ]:
pres = tweak_siena_pres(df)

In [ ]:
print(pres.head(3))

In [ ]:
print(pres.sample(3))

# Math Methods in DataFrames

#### Keeping Same Data Sources

In [ ]:
# url = 'https://github.com/mattharrison/datasets/raw/master/data/siena2018-pres.csv'

In [ ]:
df = pd.read_csv(url, index_col=0, dtype_backend='pyarrow')

In [ ]:
pres = tweak_siena_pres(df)

#### Index Alignment

In [ ]:
scores = (pres
    .loc[:, 'Background': 'Average_rank']
         )

In [ ]:
print(scores)

In [ ]:
s1 = scores.iloc[:3, :4]

print(s1)

In [ ]:
s2 = scores.iloc[1:6, :5]

print(s2)

In [ ]:
print(s1 + s2)

#### Duplicate Index Entries

In [ ]:
print(scores.iloc[:3, :4] + pd.concat([scores.iloc[1:6, :5]] * 2))

In [ ]:
pd.concat([scores.iloc[1:6, :5]] * 2).index.duplicated().any()

#### More Math

`Looking-Up on Normalization & Standarization`

In [ ]:
(scores.Imagination.sub(scores.Imagination.min())
    .div(scores.Imagination.max() - scores.Imagination.min())
)

In [ ]:
imag = scores.Imagination

In [ ]:
((imag - imag.min()) / (imag.max() - imag.min()))

In [ ]:
(scores
    .Imagination.sub(scores.Imagination.min())
    .div(scores.Imagination.max() - scores.Imagination.min())
    .describe()
)

In [ ]:
nums = scores.select_dtypes('number')

In [ ]:
print(nums.sub(nums.min()).div(nums.max()).sub(nums.min()))

In [ ]:
print(nums - nums.min()) / (nums.max() - nums.min())

In [ ]:
print(nums.sub(nums.min())
    .div(nums.max().sub(nums.min()))
    .describe()
     )

In [ ]:
class NormalizerTransformer(BaseException, TransformerMixin):
    def __init__(self):
        self.min = None
        self.max = None

    def fit(self, X, y = None):
        self.min = X.min()
        self.max = X.max()
        return self

    def transform(self, X):
        return (X - self.min) / (self.max - self.min)

In [ ]:
nt = NormalizerTransformer()

In [ ]:
print(nt.fit_transform(nums))

In [ ]:
std = nums.std()

In [ ]:
print(nums.sub(nums.mean()).div(std))

In [ ]:
print(((nums - nums.mean()) / nums.std()).describe())

In [ ]:
# Placing Scaler
ss= StandardScaler()

In [ ]:
# Fitting Scaler
results = (ss.fit_transform(nums))

In [ ]:
results

In [ ]:
ss.set_output(transform='pandas')

In [ ]:
print(ss.fit_transform(nums))

#### `PCA` Calculation in Pandas

##### Data Centered

In [ ]:
centered = nums - nums.mean()

print(centered)

##### Calculating Covariance Matrix

In [ ]:
cov = centered.cov()

print(cov)

##### Eigenvectors & Eigenvalues

In [ ]:
vals, vecs = np.linalg.eig(cov)

In [ ]:
vals

In [ ]:
vals / vals.sum()

In [ ]:
vecs[:2].round(2)

In [ ]:
idxs = pd.Series(vals).argsort()

idxs

In [ ]:
# def set_columns(df_):
#     df.columns = [f'PC{i + 1}' for i in range(len(df_.columns))]
#     return df_

In [ ]:
def set_columns(df_):
    df_ = df_.copy()
    df_.columns = [f'PC{i + 1}' for i in range(len(df_.columns))]
    return df_

In [ ]:
comps = (pd.DataFrame(vecs, index=nums.columns)
    .iloc[:, idxs[::-1]]
    .pipe(set_columns)
        )

In [ ]:
print(comps)

In [ ]:
print(centered.dot(comps))

In [ ]:
# Step 1
centered = nums - nums.mean()

In [ ]:
# Step 2 & 3
vals, vecs = np.linalg.eig(centered.cov())

In [ ]:
# Step 4
idxs = pd.Series(vals).argsort()

In [ ]:
def set_columns(df_):
    df_ = df_.copy()
    df_.columns = [f'PC{i + 1}' for i in range(len(df_.columns))]
    return df_

In [ ]:
comps = (pd.DataFrame(vecs, index=nums.columns)
    .iloc[:, idxs[::-1]]
    .pipe(set_columns)
        )

In [ ]:
# Step 5
pcas = (centered.dot(comps))

#### PCA Performed on Scikit-Learn

In [ ]:
set_config(transform_output="pandas")

In [ ]:
# Invoking PCA
pca = PCA()

In [ ]:
# Fitting PCA
print(pca.fit_transform(nums).round(2))

In [ ]:
print(pca.components_[:2].round(3))

In [ ]:
print(pd.DataFrame(pca.components_,
                  index=[f'pca{i}'for i in range(nums.shape[1])],
                  columns=nums.columns))

In [ ]:
print(pd.Series(pca.explained_variance_ratio_,
               index=[f'pca{i}' for i in range(nums.shape[1])]))

# Looping & Aggregation

### Loading Data

In [ ]:
# url = 'https://github.com/mattharrison/datasets/raw/master/data/'\
#     'siena2018-pres.csv'

In [ ]:
df = pd.read_csv(url, index_col=0, dtype_backend='pyarrow')

In [ ]:
pres = tweak_siena_pres(df)

In [ ]:
for col_name, col in pres.items():
    print(col_name, type(col))
    break

In [ ]:
for tup in pres.itertuples():
    print(tup[0], tup.President)
    break

#### Aggregations

In [ ]:
scores = (pres
    .loc[:, 'Background': 'Average_rank']
         )

In [ ]:
scores.sum(axis='columns') / len(scores.columns)

In [ ]:
print(pres.select_dtypes('number').agg(
    ['count', 'size', 'sum', lambda col: col.loc[1]]))

In [ ]:
print(pres.agg({'Luck': ['count', 'size'],
                'Overall': ['count', 'max']}))

In [ ]:
print(pres.agg(Intelligence_count=('Intelligence', 'count'),
                Intelligence_size=('Intelligence', 'size'))
     )

In [ ]:
print(pres.describe())

#### The `.apply` Method

In [ ]:
(pres
    .select_dtypes('number')
    .pipe(lambda df_: df_.max(axis='columns')
                                   - df_.min(axis='columns'))
    .rename('range')
)

In [ ]:
(pres
    .select_dtypes('number')
    .apply(lambda row: row.max() - row.min(), axis='columns')
    .rename('range')
)

In [ ]:
pres.select_dtypes('number').apply('sum')

In [ ]:
pres.select_dtypes('number').sum()

#### Optimising if Then

In [ ]:
billing_data = \
'''cancel_date,period_start,start_date,end_date,rev,sum_payments
12/1/2019,1/1/2020,12/15/2019,5/15/2020,999,50
,1/1/2020,12/15/2019,5/15/2020,999,50
,1/1/2020,12/15/2019,5/15/2020,999,1950
1/20/2020,1/1/2020,12/15/2019,5/15/2020,499,0
,1/1/2020,12/24/2019,5/24/2020,699,100
,1/1/2020,11/29/2019,4/29/2020,799,250
,1/1/2020,1/15/2020,4/29/2020,799,250'''

In [ ]:
bill_df = pd.read_csv(
    io.StringIO(billing_data),
    dtype_backend='pyarrow',
    parse_dates=[
        'cancel_date',
        'period_start',
        'start_date',
        'end_date'
    ]
)

In [ ]:
print(bill_df)

In [ ]:
def tweak_bill202(df_):
    return(df_.assign(cancel_date=pd.to_datetime(
        df_.cancel_date.replace('<NA>', ''), format='%m/%d/%Y')
                     )
          )

In [ ]:
bill_df = tweak_bill202(bill_df)

In [ ]:
def calc_unbilled_rec(vals):
    cancel_date, period_start, start_date, end_date, rev, sum_payments = vals
    if cancel_date < period_start:
        return np.nan
    if start_date < period_start and end_date > period_start:
        if rev > sum_payments:
            return rev - sum_payments
        else:
            return 0

In [ ]:
bill_df.apply(calc_unbilled_rec, axis='columns')

In [ ]:
(pd.Series(np.nan, dtype='float[pyarrow]', index=bill_df.index)
    .case_when([
        (bill_df.cancel_date < bill_df.period_start,  # 1
            np.nan),

        (((bill_df.start_date < bill_df.period_start) &  # 2
          (bill_df.end_date > bill_df.period_start) &
          (bill_df.rev > bill_df.sum_payments)),
            bill_df.rev - bill_df.sum_payments),

        (((bill_df.start_date < bill_df.period_start) &  # 3
          (bill_df.end_date > bill_df.period_start) &
          (bill_df.rev <= bill_df.sum_payments)),
            0)
    ])
)

#### Optimizing `.apply` Functions

In [ ]:
bill_100k = bill_df.sample(100_000, replace=True)

In [ ]:
%%timeit
bill_100k.apply(calc_unbilled_rec, axis='columns')

In [ ]:
def calc_unbilled_case(bill_df):
    return (pd.Series(np.nan, dtype='float[pyarrow]',
                      index=bill_df.index)
        .case_when([
            (bill_df.cancel_date < bill_df.period_start,  # 1
                np.nan),

            (((bill_df.start_date < bill_df.period_start) &  # 2
              (bill_df.end_date > bill_df.period_start) &
              (bill_df.rev > bill_df.sum_payments)),
                bill_df.rev - bill_df.sum_payments),

            (((bill_df.start_date < bill_df.period_start) &  # 3
              (bill_df.end_date > bill_df.period_start) &
              (bill_df.rev <= bill_df.sum_payments)),
                0)
        ])
    )

In [ ]:
%%timeit
calc_unbilled_case(bill_100k)

#### Optimizing `.apply` with Cython

In [ ]:
%load_ext cython

In [ ]:
%%cython

def calc_unbilled_rec_cy1(vals):
    cancel_date, period_start, start_date, end_date, rev, \
        sum_payments = vals

    if cancel_date < period_start:
        return float('nan')

    if start_date < period_start and end_date > period_start:
        if rev > sum_payments:
            return rev - sum_payments
        else:
            return 0

In [ ]:
calc_unbilled_rec_cy1(bill_df.iloc[0])

In [ ]:
%%timeit
bill_100k.apply(calc_unbilled_rec_cy1, axis='columns')

In [ ]:
%%cython

import cython
import numpy as np
cimport numpy as np


@cython.wraparound(False)
cpdef calc_unbilled_cy(np.ndarray[np.int64_t] cancel_date,
                       np.ndarray[np.int64_t] period_start,
                       np.ndarray[np.int64_t] start_date,
                       np.ndarray[np.int64_t] end_date,
                       np.ndarray[np.int64_t] rev,
                       np.ndarray[np.int64_t] sum_payments):

    cdef np.ndarray[np.float64_t] results = np.full(
        rev.shape[0],
        np.nan,
        dtype=np.float64
    )

    cdef long cd, pd, sd, ed

    for i in range(rev.shape[0]):

        cd = cancel_date[i]
        ps = period_start[i]
        sd = start_date[i]
        ed = end_date[i]

        if cd > 0 and cd < ps:
            continue

        elif sd < ps < ed:

            if rev[i] > sum_payments[i]:
                results[i] = rev[i] - sum_payments[i]

            else:
                results[i] = 0

    return results

In [ ]:
%%timeit
calc_unbilled_cy(*(ser.astype(int).to_numpy()
                   for name, ser in bill_100k.items()))

In [ ]:
%%cython

import cython
import numpy as np


@cython.wraparound(False)
def calc_unbilled_cy2(cancel_date: np.ndarray[np.int64_t],
                      period_start: np.ndarray[np.int64_t],
                      start_date: np.ndarray[np.int64_t],
                      end_date: np.ndarray[np.int64_t],
                      rev: np.ndarray[np.int64_t],
                      sum_payments: np.ndarray[np.int64_t]
                      ) -> np.ndarray[np.float64_t]:

    results: np.ndarray[np.float64_t] = np.full(
        rev.shape[0],
        np.nan,
        dtype=np.float64
    )

    i: cython.int

    for i in range(rev.shape[0]):

        cd: cython.long = cancel_date[i]
        ps: cython.long = period_start[i]
        sd: cython.long = start_date[i]
        ed: cython.long = end_date[i]

        if cd > 0 and cd < ps:
            continue

        elif sd < ps < ed:

            if rev[i] > sum_payments[i]:
                results[i] = rev[i] - sum_payments[i]

            else:
                results[i] = 0

    return results

In [ ]:
%%timeit
calc_unbilled_cy2(*(ser.astype(int).to_numpy()
                    for name, ser in bill_100k.items()))

In [ ]:
%%cython

import cython
import numpy as np


@cython.wraparound(False)
def calc_unbilled_cy3(cancel_date: cython.long[:],
                      period_start: cython.long[:],
                      start_date: cython.long[:],
                      end_date: cython.long[:],
                      rev: cython.long[:],
                      sum_payments: cython.long[:]
                      ) -> cython.double[:]:

    results: cython.double[:] = np.full(
        rev.shape[0],
        np.nan,
        dtype=np.float64
    )

    i: cython.int

    for i in range(rev.shape[0]):

        cd: cython.long = cancel_date[i]
        ps: cython.long = period_start[i]
        sd: cython.long = start_date[i]
        ed: cython.long = end_date[i]

        if cd > 0 and cd < ps:
            continue

        elif sd < ps < ed:

            if rev[i] > sum_payments[i]:
                results[i] = rev[i] - sum_payments[i]

            else:
                results[i] = 0

    return results

In [ ]:
%%timeit

calc_unbilled_cy3(
    *(ser.astype(int).to_numpy(copy=True)
      for name, ser in bill_100k.items())
)

In [ ]:
%%cython

import cython
import numpy as np


@cython.boundscheck(False)
@cython.wraparound(False)
def calc_unbilled_cy4(cancel_date: cython.long[:],
                      period_start: cython.long[:],
                      start_date: cython.long[:],
                      end_date: cython.long[:],
                      rev: cython.long[:],
                      sum_payments: cython.long[:]
                      ) -> cython.double[:]:

    results: cython.double[:] = np.full(
        rev.shape[0],
        np.nan,
        dtype=np.float64
    )

    i: cython.int

    assert len(period_start) == len(start_date)

    for i in range(rev.shape[0]):

        cd: cython.long = cancel_date[i]
        ps: cython.long = period_start[i]
        sd: cython.long = start_date[i]
        ed: cython.long = end_date[i]

        if cd > 0 and cd < ps:
            continue

        elif sd < ps < ed:

            if rev[i] > sum_payments[i]:
                results[i] = rev[i] - sum_payments[i]

            else:
                results[i] = 0

    return results

In [ ]:
%%timeit

calc_unbilled_cy4(
    *(ser.astype("int64").to_numpy(copy=True)
      for name, ser in bill_100k.items())
)

In [ ]:
%%cython

import cython
from cython.parallel import prange
import numpy as np


@cython.boundscheck(False)
@cython.wraparound(False)
def calc_unbilled_cy5(cancel_date: cython.long[:],
                      period_start: cython.long[:],
                      start_date: cython.long[:],
                      end_date: cython.long[:],
                      rev: cython.long[:],
                      sum_payments: cython.long[:]
                      ) -> cython.double[:]:

    results: cython.double[:] = np.full(
        rev.shape[0],
        np.nan,
        dtype=np.float64
    )

    i: cython.int

    assert len(period_start) == len(start_date)

    for i in prange(rev.shape[0], nogil=True):

        cd: cython.long = cancel_date[i]
        ps: cython.long = period_start[i]
        sd: cython.long = start_date[i]
        ed: cython.long = end_date[i]

        if cd > 0 and cd < ps:
            continue

        elif sd < ps < ed:

            if rev[i] > sum_payments[i]:
                results[i] = rev[i] - sum_payments[i]

            else:
                results[i] = 0

    return results

In [ ]:
%%timeit

calc_unbilled_cy5(
    *(ser.astype(int).to_numpy(copy=True)
      for name, ser in bill_100k.items())
)

#### Optimization with Numba

In [ ]:
@jit
def calc_unbilled_numba(cancel_date, period_start, start_date,
                        end_date, rev, sum_payments):

    results = np.full(rev.shape[0], np.nan, dtype=np.float64)

    for i in range(rev.shape[0]):
        cd = cancel_date[i]
        ps = period_start[i]
        sd = start_date[i]
        ed = end_date[i]

        if cd > 0 and cd < ps:
            results[i] = np.nan

        elif sd < ps < ed:
            if rev[i] > sum_payments[i]:
                results[i] = rev[i] - sum_payments[i]
            else:
                results[i] = 0

    return results

In [ ]:
%%timeit

calc_unbilled_numba(
    *(ser.astype(int).to_numpy()
      for name, ser in bill_100k.items())
)

In [ ]:
# print(bill_df
#     .assign(
#         cy=calc_unbilled_cy(
#             *(ser.astype(int).to_numpy()
#               for name, ser in bill_df.items())
#         ),
#         nb=calc_unbilled_numba(
#             *(ser.astype(int).to_numpy()
#               for name, ser in bill_df.items())
#         ),
#         np=calc_unbilled_np(bill_df),
#         py=bill_df.apply(calc_unbilled_rec, axis='columns')
#     )
# )

# Columns Types, `.assign` and Memory Usage

#### Keeping Same Dataset

In [ ]:
df = pd.read_csv(url, index_col=0, dtype_backend='pyarrow', engine='pyarrow')

In [ ]:
pres = tweak_siena_pres(df)

#### Memory Usage

In [ ]:
df.memory_usage(deep=True)

In [ ]:
pres.memory_usage(deep=True)

In [ ]:
df.memory_usage(deep=True).sum(), pres.memory_usage(deep=True).sum()

In [ ]:
pres.info(memory_usage='deep')

# Creating & Updating Columns

### Loading Data

In [ ]:
url = 'https://github.com/mattharrison/datasets/raw/master/data/2020-jetbrains-python-survey.csv'

In [ ]:
jb = pd.read_csv(url, dtype_backend='pyarrow', engine='pyarrow')

In [ ]:
print(jb)

In [ ]:
counter = collections.defaultdict(list)

In [ ]:
for col in sorted(jb.columns):
    period_count = col.count('.')
    if period_count >= 2:
        part_end = 2
    else:
        part_end = 1
    parts = col.split('.')[:part_end]
    counter['.'.join(parts)].append(col)

In [ ]:
uniq_cols = []

In [ ]:
for cols in counter.values():
    if len(cols) == 1:
        uniq_cols.extend(cols)

In [ ]:
uniq_cols

In [ ]:
(jb
    [uniq_cols]
    .rename(columns=lambda c: c.replace('.', '_'))
    .age
    .value_counts(dropna=False)
)

In [ ]:
(jb
    [uniq_cols]
    .rename(columns=lambda c: c.replace('.', '_'))
    .age
    .str.slice(0, 2)
    .replace('', np.nan)
    .astype('int8[pyarrow]')
)

#### The `.assign` Method

In [ ]:
jb2 = jb[uniq_cols]

In [ ]:
age_slice = jb.age.str.slice(0, 2)

In [ ]:
age_float = age_slice.astype(float)

In [ ]:
age_int = age_float.astype('Int64')

In [ ]:
jb2['age'] = age_int

In [ ]:
print(jb
    [uniq_cols]
    .rename(columns=lambda c: c.replace('.', '_'))
    .assign(age=lambda df_: df_.age
        .str.slice(0, 2)
        .replace('', np.nan)
        .astype('int8[pyarrow]'))
     )

#### More Column Cleanup

In [ ]:
(jb
    [uniq_cols]
    .rename(columns=lambda c: c.replace('.', '_'))
    .assign(age=lambda df_: df_.age.str.slice(0, 2)
                .astype('int8[pyarrow]'),
            are_you_datascientist=lambda df_: df_.are_you_datascientist
                .replace({'Yes': '1', 'No': '0', '': '0', 'Other': '0'})
                .astype('bool[pyarrow]')
    )
    .are_you_datascientist
)

In [ ]:
(jb
    [uniq_cols]
    .rename(columns=lambda c: c.replace('.', '_'))
    .assign(age=lambda df_: df_.age.str.slice(0, 2)
                .astype('int8[pyarrow]'),
            are_you_datascientist=lambda df_: df_.are_you_datascientist
                .replace({'Yes': '1', 'No': '0', '': '0', 'Other': '0'})
                .astype('bool[pyarrow]')
    )
    .company_size
    .value_counts(dropna=False)
)

In [ ]:
(jb
    [uniq_cols]
    .rename(columns=lambda c: c.replace('.', '_'))
    .assign(age=lambda df_: df_.age.str.slice(0, 2)
                .astype('int8[pyarrow]'),
            are_you_datascientist=lambda df_: df_.are_you_datascientist
                .replace({'Yes': '1', 'No': '0', '': '0', 'Other': '0'})
                .astype('bool[pyarrow]')
    )
    .company_size
    .replace({'Just me': '1', '': pd.NA,
              'Not sure': pd.NA, 'More than 5,000': '5000',
              '2–10': '2', '11–50': '11', '51–500': '51',
              '501–1,000': '501', '1,001–5,000': '1001'})
    .astype('int64[pyarrow]')
    .value_counts()
)

In [ ]:
company_size=lambda df_: df_.company_size.replace(
    {'Just me': '1', '': pd.NA,
     'Not sure': pd.NA, 'More than 5,000': '5000',
     '2–10': '2', '11–50': '11', '51–500': '51',
     '501–1,000': '501', '1,001–5,000': '1001'}
).astype('int64[pyarrow]')

In [ ]:
jb2 = (jb
    [uniq_cols]
    .rename(columns=lambda c: c.replace('.', '_'))
    .assign(age=lambda df_: df_.age.str.slice(0, 2)
                .astype('int8[pyarrow]'),

            are_you_datascientist=lambda df_: df_.are_you_datascientist
                .replace({'Yes': '1', 'No': '0', '': '0', 'Other': '0'})
                .astype('bool[pyarrow]'),

            company_size=lambda df_: df_.company_size.replace(
                {'Just me': '1', '': pd.NA,
                 'Not sure': pd.NA, 'More than 5,000': '5000',
                 '2-10': '2', '2–10': '2',
                 '11-50': '11', '11–50': '11',
                 '51-500': '51', '51–500': '51',
                 '501-1,000': '501', '501–1,000': '501',
                 '1,001-5,000': '1001', '1,001–5,000': '1001'}
            ).astype('int64[pyarrow]'),

            country_live=lambda df_: df_.country_live.astype('category'),

            employment_status=lambda df_: df_.employment_status
                .fillna('Other').astype('category'),

            is_python_main=lambda df_: df_.is_python_main
                .astype('category'),

            team_size=lambda df_: df_.team_size
                .replace('More than 40 people', '41')
                .str.extract(r'(?P<team_size>\d+)')
                .where(df_.company_size != 1, '1')
                .replace('', pd.NA)
                .astype('int8[pyarrow]'),

            years_of_coding=lambda df_: df_.years_of_coding
                .replace('Less than 1 year', '.5')
                .str.extract(r'(?P<years_of_coding>\.?\d+)')
                .astype('float64[pyarrow]'),

            python_years=lambda df_: df_.python_years
                .replace('Less than 1 year', '.5')
                .str.extract(r'(?P<python_years>\.?\d+)')
                .astype('float64[pyarrow]'),

            python3_ver=lambda df_: df_.python3_version_most
                .str.replace('_', '.')
                .str.extract(r'(?P<python3_ver>\d\.\d)'),

            use_python_most=lambda df_: df_.use_python_most
                .fillna('Unknown')
    )
    .drop(columns=['python2_version_most'])
)

In [ ]:
print(jb2)

In [ ]:
jb2.columns

In [ ]:
jb2 = (jb
    [uniq_cols]
    .rename(columns=lambda c: c.replace('.', '_'))
    .assign(age=lambda df_: df_.age.str.slice(0, 2)
                .astype('int8[pyarrow]'),

            are_you_datascientist=lambda df_: df_.are_you_datascientist
                .replace({'Yes': '1', 'No': '0', '': '0', 'Other': '0'})
                .astype('bool[pyarrow]'),

            company_size=lambda df_: df_.company_size.replace(
                {'Just me': '1', '': pd.NA,
                 'Not sure': pd.NA, 'More than 5,000': '5000',
                 '2-10': '2', '2–10': '2',
                 '11-50': '11', '11–50': '11',
                 '51-500': '51', '51–500': '51',
                 '501-1,000': '501', '501–1,000': '501',
                 '1,001-5,000': '1001', '1,001–5,000': '1001'}
            ).astype('int64[pyarrow]'),

            country_live=lambda df_: df_.country_live.astype('category'),

            employment_status=lambda df_: df_.employment_status
                .fillna('Other').astype('category'),

            is_python_main=lambda df_: df_.is_python_main
                .astype('category'),

            team_size=lambda df_: df_.team_size
                .replace('More than 40 people', '41')
                .str.extract(r'(?P<team_size>\d+)')
                .where(df_.company_size != 1, '1')
                .replace('', pd.NA)
                .astype('int8[pyarrow]'),

            years_of_coding=lambda df_: df_.years_of_coding
                .replace('Less than 1 year', '.5')
                .str.extract(r'(?P<years_of_coding>\.?\d+)')
                .astype('float64[pyarrow]'),

            python_years=lambda df_: df_.python_years
                .replace('Less than 1 year', '.5')
                .str.extract(r'(?P<python_years>\.?\d+)')
                .astype('float64[pyarrow]'),

            python3_ver=lambda df_: df_.python3_version_most
                .str.replace('_', '.')
                .str.extract(r'(?P<python3_ver>\d\.\d)'),

            use_python_most=lambda df_: df_.use_python_most
                .fillna('Unknown')
    )
    .loc[:, ['age', 'are_you_datascientist', 'company_size',
             'country_live', 'employment_status',
             'first_learn_about_main_ide', 'how_often_use_main_ide',
             'ide_main', 'is_python_main', 'job_team',
             'main_purposes', 'missing_features_main_ide',
             'nps_main_ide', 'python_years',
             'python3_version_most', 'several_projects',
             'team_size', 'use_python_most',
             'years_of_coding', 'python3_ver']]
)

In [ ]:
def limit_index_length(df, max_length):
    return df.rename(index=lambda x: x[:max_length])

In [ ]:
(jb2
    .query('team_size.isna()')
    .employment_status
    .value_counts(dropna=False)
    .pipe(limit_index_length, max_length=40)
)

In [ ]:
cb.__version__

In [ ]:
(df
    .select_dtypes(['object', 'category', 'string'])
    .astype('str')
    .fillna('')
)

In [ ]:
{col: df[col].astype(str).fillna('')
 for col in df.select_dtypes(['object',
                             'category', 'string'])}

In [ ]:
def prep_for_ml(df):
    # remove pandas types
    return (df
        .assign(**df.select_dtypes(['number', 'bool'])
            .astype('float[pyarrow]').astype(float),
            **{col: df[col].astype(str).fillna('')
               for col in df.select_dtypes(['object',
                                             'category', 'string'])})
    )

In [ ]:
# def predict_col(df, col):
#     df = prep_for_ml(df)
#     missing = df.query(f'~{col}.isna()')
#     cat_idx = [i for i, typ in enumerate(df.drop(columns=[col]).dtypes)
#                if str(typ) == 'object']

#     X = (missing
#         .drop(columns=[col])
#         .values
#     )
#     y = missing[col]

#     model = cb.CatBoostRegressor(iterations=20, cat_features=cat_idx)
#     model.fit(X, y, cat_features=cat_idx)
#     pred = model.predict(df.drop(columns=[col]))
#     return df[col].where(~df[col].isna(), pred)

In [ ]:
from pandas.api.types import is_numeric_dtype

def predict_col(df, col):
    df = prep_for_ml(df)

    missing = df.query(f'~{col}.isna()')

    X = missing.drop(columns=[col])
    y = missing[col]

    cat_cols = [
        col_name for col_name in X.columns
        if not is_numeric_dtype(X[col_name])
    ]

    model = cb.CatBoostRegressor(
        iterations=20,
        cat_features=cat_cols
    )

    model.fit(X, y, cat_features=cat_cols)

    pred = model.predict(df.drop(columns=[col]))

    return df[col].where(~df[col].isna(), pred)

In [ ]:
jb2 = (jb
    [uniq_cols]
    .rename(columns=lambda c: c.replace('.', '_'))
    .assign(age=lambda df_: df_.age.str.slice(0, 2)
                .astype('int8[pyarrow]'),

            are_you_datascientist=lambda df_: df_.are_you_datascientist
                .replace({'Yes': '1', 'No': '0', '': '0', 'Other': '0'})
                .astype('bool[pyarrow]'),

            company_size=lambda df_: df_.company_size.replace(
                {'Just me': '1', '': pd.NA,
                 'Not sure': pd.NA, 'More than 5,000': '5000',
                 '2–10': '2', '11–50': '11', '51–500': '51',
                 '501–1,000': '501', '1,001–5,000': '1001'}
            ).astype('int64[pyarrow]'),

            country_live=lambda df_: df_.country_live.astype('category'),

            employment_status=lambda df_: df_.employment_status
                .fillna('Other').astype('category'),

            is_python_main=lambda df_: df_.is_python_main
                .astype('category'),

            team_size=lambda df_: df_.team_size
                .replace('More than 40 people', '41')
                .str.extract(r'(?P<team_size>\d+)')
                .where(df_.company_size != 1, '1')
                .replace('', pd.NA)
                .astype('int8[pyarrow]'),

            years_of_coding=lambda df_: df_.years_of_coding
                .replace('Less than 1 year', '.5')
                .str.extract(r'(?P<years_of_coding>\.?\d+)')
                .astype('float64[pyarrow]'),

            python_years=lambda df_: df_.python_years
                .replace('Less than 1 year', '.5')
                .str.extract(r'(?P<python_years>\.?\d+)')
                .astype('float64[pyarrow]'),

            python3_ver=lambda df_: df_.python3_version_most
                .str.replace('_', '.')
                .str.extract(r'(?P<python3_ver>\d\.\d)'),

            use_python_most=lambda df_: df_.use_python_most
                .fillna('Unknown')
    )
    .assign(
        team_size=lambda df_: predict_col(df_, 'team_size')
            .astype(int)
    )
    .loc[:, ['age', 'are_you_datascientist', 'company_size',
             'country_live', 'employment_status',
             'first_learn_about_main_ide', 'how_often_use_main_ide',
             'ide_main', 'is_python_main', 'job_team',
             'main_purposes', 'missing_features_main_ide',
             'nps_main_ide', 'python_years',
             'python3_version_most', 'several_projects',
             'team_size', 'use_python_most',
             'years_of_coding', 'python3_ver']]
)

In [ ]:
print(jb2)

In [ ]:
def get_uniq_cols(jb):
    counter = collections.defaultdict(list)

    for col in sorted(jb.columns):

        period_count = col.count('.')

        if period_count >= 2:
            part_end = 2
        else:
            part_end = 1

        parts = col.split('.')[:part_end]

        counter['.'.join(parts)].append(col)

    uniq_cols = []

    for cols in counter.values():

        if len(cols) == 1:
            uniq_cols.extend(cols)

    return uniq_cols

In [ ]:
def prep_for_ml(df):

    # remove pandas/pyarrow types
    return (
        df
        .assign(

            **df.select_dtypes(['number', 'bool'])
                .astype('float[pyarrow]')
                .astype(float),

            **{
                col: df[col]
                    .astype(str)
                    .fillna('')

                for col in df.select_dtypes(
                    ['object', 'category', 'string']
                )
            }
        )
    )

In [ ]:
from pandas.api.types import is_numeric_dtype

def predict_col(df, col):
    df = prep_for_ml(df)

    missing = df.query(f'~{col}.isna()')

    X = missing.drop(columns=[col])
    y = missing[col]

    cat_cols = [
        col_name for col_name in X.columns
        if not is_numeric_dtype(X[col_name])
    ]

    model = cb.CatBoostRegressor(
        iterations=20,
        cat_features=cat_cols,
        verbose=False
    )

    model.fit(X, y, cat_features=cat_cols)

    pred = model.predict(df.drop(columns=[col]))

    return df[col].where(~df[col].isna(), pred)

In [ ]:
def tweak_jb(jb):

    uniq_cols = get_uniq_cols(jb)

    return (
        jb
        [uniq_cols]

        .rename(columns=lambda c: c.replace('.', '_'))

        .assign(

            age=lambda df_: (
                df_.age
                    .str.slice(0, 2)
                    .astype('int8[pyarrow]')
            ),

            are_you_datascientist=lambda df_: (
                df_.are_you_datascientist
                    .replace({
                        'Yes': '1',
                        'No': '0',
                        '': '0',
                        'Other': '0'
                    })
                    .astype('bool[pyarrow]')
            ),

            company_size=lambda df_: (
                df_.company_size
                    .replace({
                        'Just me': '1',
                        '': pd.NA,
                        'Not sure': pd.NA,
                        'More than 5,000': '5000',
                        '2–10': '2',
                        '11–50': '11',
                        '51–500': '51',
                        '501–1,000': '501',
                        '1,001–5,000': '1001'
                    })
                    .astype('int64[pyarrow]')
            ),

            country_live=lambda df_: (
                df_.country_live
                    .astype('category')
            ),

            employment_status=lambda df_: (
                df_.employment_status
                    .fillna('Other')
                    .astype('category')
            ),

            is_python_main=lambda df_: (
                df_.is_python_main
                    .astype('category')
            ),

            team_size=lambda df_: (
                df_.team_size
                    .replace('More than 40 people', '41')
                    .str.extract(r'(?P<team_size>\d+)')
                    .where(df_.company_size != 1, '1')
                    .replace('', pd.NA)
                    .astype('int8[pyarrow]')
            ),

            years_of_coding=lambda df_: (
                df_.years_of_coding
                    .replace('Less than 1 year', '.5')
                    .str.extract(
                        r'(?P<years_of_coding>\.?\d+)'
                    )
                    .astype('float64[pyarrow]')
            ),

            python_years=lambda df_: (
                df_.python_years
                    .replace('Less than 1 year', '.5')
                    .str.extract(
                        r'(?P<python_years>\.?\d+)'
                    )
                    .astype('float64[pyarrow]')
            ),

            python3_ver=lambda df_: (
                df_.python3_version_most
                    .str.replace('_', '.')
                    .str.extract(
                        r'(?P<python3_ver>\d\.\d)'
                    )
            ),

            use_python_most=lambda df_: (
                df_.use_python_most
                    .fillna('Unknown')
            )
        )

        .assign(
            team_size=lambda df_: (
                predict_col(df_, 'team_size')
                    .astype(int)
            )
        )

        .loc[:, [
            'age',
            'are_you_datascientist',
            'company_size',
            'country_live',
            'employment_status',
            'first_learn_about_main_ide',
            'how_often_use_main_ide',
            'ide_main',
            'is_python_main',
            'job_team',
            'main_purposes',
            'missing_features_main_ide',
            'nps_main_ide',
            'python_years',
            'python3_version_most',
            'several_projects',
            'team_size',
            'use_python_most',
            'years_of_coding',
            'python3_ver'
        ]]
    )

In [ ]:
url = (
    'https://github.com/mattharrison/datasets/raw/master/data/'
    '2020-jetbrains-python-survey.csv'
)

In [ ]:
jb = pd.read_csv(
    url,
    dtype_backend='pyarrow',
    engine='pyarrow'
)

In [ ]:
jb2 = tweak_jb(jb)

# Dealing with Missing & Duplicated Data

#### Placing & Keeping Same Data (`siena.df`)

In [ ]:
def tweak_siena_pres(df):

    def int64_to_uint8(df_):

        cols = df_.select_dtypes('int64')

        return (
            df_
                .astype({
                    col: 'uint8[pyarrow]'
                    for col in cols
                })
        )

    return (

        df

        .rename(columns={'Seq.': 'Seq'})  # 1

        .rename(columns={
            k: v.replace(' ', '_')
            for k, v in {

                'Bg': 'Background',
                'PL': 'Party leadership',
                'CAb': 'Communication ability',
                'RC': 'Relations with Congress',
                'CAp': 'Court appointments',
                'HE': 'Handling of economy',
                'L': 'Luck',
                'AC': 'Ability to compromise',
                'WR': 'Willing to take risks',
                'EAp': 'Executive appointments',
                'OA': 'Overall ability',
                'Im': 'Imagination',
                'DA': 'Domestic accomplishments',
                'Int': 'Integrity',
                'EAb': 'Executive ability',
                'FPA': 'Foreign policy accomplishments',
                'LA': 'Leadership ability',
                'IQ': 'Intelligence',
                'AM': 'Avoid crucial mistakes',
                'EV': "Experts' view",
                'O': 'Overall'

            }.items()
        })  # 2

        .astype({'Party': 'category'})

        .pipe(int64_to_uint8)  # 3

        .assign(

            Average_rank=lambda df_: (
                df_
                    .select_dtypes('uint8')
                    .sum(axis=1)
                    .rank(method='dense')
                    .astype('uint8[pyarrow]')
            ),  # 4

            Quartile=lambda df_: (
                pd.qcut(
                    df_.Average_rank,
                    4,
                    labels='1st 2nd 3rd 4th'.split()
                )
            )
        )
    )

In [ ]:
url = (
    'https://github.com/mattharrison/datasets/raw/master/data/'
    'siena2018-pres.csv')

In [ ]:
df = pd.read_csv(url, index_col=0, dtype_backend='pyarrow')

pres = tweak_siena_pres(df)

In [ ]:
print(pres.isna())

In [ ]:
print(pres[pres.Integrity.isna()])

In [ ]:
# The query Approach

print(pres.query('Integrity.isna()'))

In [ ]:
pres.isna().sum()

In [ ]:
pres.isna().mean()

#### Duplicates

In [ ]:
print(pres.drop_duplicates())

In [ ]:
print(pres.drop_duplicates(subset='Party'))

In [ ]:
print(pres.drop_duplicates(subset='Party', keep='last'))

In [ ]:
print(pres.drop_duplicates(subset='Party', keep=False))

In [ ]:
print(pres
    .assign(first_in_party_seq=lambda df_: df_.Party != df_.Party.shift(1))
    .query('first_in_party_seq')
     )

# Sorting Columns & Indexes

#### Sorting Columns

In [ ]:
print(pres.sort_values(by='Party'))

In [ ]:
print(pres
    .sort_values(by=['Party', 'Average_rank'],
                       ascending=[True, False])
     )

In [ ]:
print(pres
    .President
    .str.split()
     )

In [ ]:
print(pres
    .President
    .str.split()
    .apply(lambda val: val[-1])
     )

In [ ]:
print(pres
    .President
    .astype(str)
    .str.split()
    .str[-1]
     )

In [ ]:
print(pres
    .sort_values(by='President', key=lambda name_ser: name_ser.astype(str)
        .str.split()
        .str[-1])
     )

#### Sorting Column Order

In [ ]:
print(pres
    .sort_index(axis='columns')
     )

#### Setting & Sorting The Index

In [ ]:
print(pres
    .set_index('President')
    .sort_index()
     )

In [ ]:
(pres
    .set_index('Party')
    .loc['Democratic': 'Republican']
)

In [ ]:
print(pres
    .set_index('Party')
    .sort_index()
    .loc['Democratic':'Republican']
     )

# Filtering & Indexing Operations

#### Renaming Index

In [ ]:
def name_to_initial(val):
    names = val.split()
    return ' '.join([f'{names[0][0]}.', *names[1:]])

In [ ]:
print(pres
    .set_index('President')
    .rename(name_to_initial)
     )

#### Resetting The Index

In [ ]:
print(pres
    .set_index('President')
    .reset_index()
     )

#### Dataframe Indexing, Filtering & Querying

In [ ]:
lt10 = pres.Average_rank < 10

In [ ]:
print([lt10])

In [ ]:
print(pres[lt10 & (pres.Party == 'Republican')])

In [ ]:
print(pres[(pres.Average_rank < 10) & (pres.Party == 'Republican')])

In [ ]:
print(pres.query('Average_rank < 10 and Party == "Republican"'))

In [ ]:
lt10 = pres.Average_rank < 10

In [ ]:
print(pres.query('@lt10 and Party == "Republican"'))

#### Indexing by Position

In [ ]:
pres.iloc[1]

In [ ]:
print(pres.iloc[[1]])

In [ ]:
print(pres.iloc[[0, 5, 10]])

In [ ]:
print(pres.iloc[0:11:5])

In [ ]:
print(pres.iloc[[0, 5, 10]])

In [ ]:
print(pres.iloc[lambda df: [0, 5, 10]])

In [ ]:
pres.iloc[[0, 5, 10], 1]

In [ ]:
print(pres.iloc[[0, 5, 10], [1]])

In [ ]:
print(pres.iloc[:, [1, 2]])

In [ ]:
print(pres.iloc[:, 1:3])

#### Indexing by Name

In [ ]:
pres.loc[1:5]

In [ ]:
print(pres.loc['1':'5'])

In [ ]:
print(pres.loc[1:5])

In [ ]:
print(pres.iloc[1:5])

In [ ]:
print(pres
    .set_index('Party')
    .loc['Whig']
     )

In [ ]:
print(pres
    .set_index('Party')
    .loc[['Whig']]
     )

In [ ]:
(pres
    .set_index('Party')
    .loc['Federalist']
)

In [ ]:
print(pres
    .set_index('Party')
    .loc[['Federalist']]
     )

In [ ]:
(pres
    .set_index('Party')
    .loc['Democratic':'Independent']
)

In [ ]:
print(pres
    .set_index('Party')
    .sort_index()
    .loc['Democratic':'Independent']
     )

In [ ]:
print(pres
    .set_index('President')
    .sort_index()
    .loc['C':'Thomas Jefferson', 'Party':'Integrity']
     )

In [ ]:
(pres
    .set_index('Party')
    .sort_index()
    .loc['D':'J']
)

In [ ]:
string_pa = pd.ArrowDtype(pa.string())

In [ ]:
print(pres
    .assign(Party=pres.Party.astype(string_pa))
    .set_index('Party')
    .sort_index().loc['D':'J']
     )

In [ ]:
print(pres
    .set_index('President')
    .sort_index()
    .sort_index(axis='columns')
    .loc['C':'Thomas Jefferson', 'B':'D']
     )

#### Filtering with Functions & `.loc`

In [ ]:
print(pres
    .loc[pres.Average_rank < 10, lambda df_: df_.columns[:3]]
     )

# Plotting with Dataframes

#### Lines Plots

In [ ]:
df = pd.read_csv(url, index_col=0)

In [ ]:
pres = tweak_siena_pres(df)

In [ ]:
pres.plot().legend(bbox_to_anchor=(1, 1))

plt.grid(True)
plt.show()